# 02 - Bag-of-Words, TF-IDF vs Embeddings

---

## Learning Objectives

By the end of this notebook you will be able to:

- Clearly distinguish **"Bagging" (Bootstrap Aggregating)** from **"Bag-of-Words" (BoW)**.
- Build a **Bag-of-Words** representation and understand its document-term matrix.
- Compute **TF-IDF** weights and explain why they improve on raw counts.
- Describe **word embeddings** (Word2Vec, GloVe) as dense, learned representations.
- Compare BoW, TF-IDF, and Embeddings on a text classification task.

## Prerequisites

- Basic Python and NumPy.
- Completed Notebook 01 (text preprocessing basics).
- Familiarity with sklearn's API (`fit`, `transform`).

## Table of Contents

1. [IMPORTANT: "Bagging" vs "Bag-of-Words" Clarification](#1-important-bagging-vs-bag-of-words-clarification)
2. [Bag-of-Words (BoW)](#2-bag-of-words-bow)
3. [TF-IDF](#3-tf-idf)
4. [Word Embeddings (Overview)](#4-word-embeddings-overview)
5. [Comparison Table](#5-comparison-table)
6. [Code: BoW with CountVectorizer](#6-code-bow-with-countvectorizer)
7. [Code: TF-IDF with TfidfVectorizer](#7-code-tf-idf-with-tfidfvectorizer)
8. [Code: Text Classification Comparison](#8-code-text-classification-comparison)
9. [Common Mistakes & Debugging Tips](#9-common-mistakes--debugging-tips)
10. [Exercises](#10-exercises)
11. [Exercise Solutions](#11-exercise-solutions)

In [ ]:
# ---- Environment setup ----
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

set_global_seed(42)

import numpy as np
import pandas as pd
from collections import Counter

print("Setup complete.")

## 1. IMPORTANT: "Bagging" vs "Bag-of-Words" Clarification

These two concepts are **completely different** despite similar-sounding names. This section clears up a common source of confusion.

---

### "Bagging" = Bootstrap Aggregating (Ensemble ML Method)

**Bagging** is short for **B**ootstrap **Agg**regat**ing**, an ensemble machine learning technique:

- **How it works:** Train multiple models on different **bootstrap samples** (random subsets with replacement) of the training data, then **aggregate** their predictions (majority vote for classification, average for regression).
- **Most famous example:** Random Forest = Bagging + Decision Trees.
- **Purpose:** Reduce variance and prevent overfitting.
- **Domain:** General machine learning, not specific to NLP.

```
Bagging (Bootstrap Aggregating):
    Dataset --> [Bootstrap Sample 1] --> Model 1 --\
            --> [Bootstrap Sample 2] --> Model 2 ---+--> Aggregate --> Final Prediction
            --> [Bootstrap Sample 3] --> Model 3 --/
```

### "Bag-of-Words" (BoW) = NLP Text Representation Method

**Bag-of-Words** is a text representation technique in NLP:

- **How it works:** Represent a document as a **vector of word counts** (or frequencies), ignoring word order.
- **"Bag"** metaphor: Imagine throwing all the words of a document into a bag -- you know *which* words are there and *how many*, but you lose their order.
- **Purpose:** Convert text to numerical features for machine learning.
- **Domain:** NLP / text processing.

```
Bag-of-Words:
    "the cat sat on the mat" --> {the: 2, cat: 1, sat: 1, on: 1, mat: 1}
```

### Side-by-Side Comparison

| Aspect | **Bagging** (Bootstrap Aggregating) | **Bag-of-Words** (BoW) |
|--------|-------------------------------------|------------------------|
| **What it is** | Ensemble learning technique | Text representation method |
| **Domain** | General ML | NLP |
| **Input** | Any dataset | Text documents |
| **Output** | Aggregated model predictions | Numerical feature vectors |
| **Key idea** | Train multiple models on data subsets | Count word occurrences |
| **Example** | Random Forest | Document-term matrix |
| **Reduces** | Variance (overfitting) | Text to numbers |

> **Bottom line:** If someone says "bagging" in an NLP context, they almost certainly mean **Bag-of-Words** (BoW). If they say "bagging" in a general ML context, they mean **Bootstrap Aggregating**. The rest of this notebook focuses entirely on **Bag-of-Words** and related NLP text representations.

## 2. Bag-of-Words (BoW)

### Concept

BoW represents each document as a **fixed-length vector** where each dimension corresponds to a word in the vocabulary:

$$\mathbf{x}_d = [\text{count}(w_1, d), \text{count}(w_2, d), \ldots, \text{count}(w_V, d)]$$

where $V$ is the vocabulary size and $\text{count}(w_i, d)$ is the number of times word $w_i$ appears in document $d$.

### Document-Term Matrix

For a corpus of $D$ documents with vocabulary size $V$, the document-term matrix is $D \times V$:

$$M_{ij} = \text{count of word } j \text{ in document } i$$

### Limitations

- **No word order:** "dog bites man" and "man bites dog" have identical BoW vectors.
- **No semantics:** "happy" and "joyful" are treated as completely unrelated.
- **High dimensionality:** vocabulary can be 100K+ words.
- **Sparse:** most entries are zero.

In [ ]:
# ---- Bag-of-Words from scratch ----

def build_bow(documents: list) -> tuple:
    """Build a Bag-of-Words matrix from scratch.
    
    Returns:
        (matrix, vocabulary): BoW matrix and word-to-index mapping.
    """
    # Build vocabulary
    vocab = {}
    for doc in documents:
        for word in doc.lower().split():
            if word not in vocab:
                vocab[word] = len(vocab)
    
    # Build matrix
    matrix = np.zeros((len(documents), len(vocab)), dtype=int)
    for i, doc in enumerate(documents):
        for word in doc.lower().split():
            matrix[i, vocab[word]] += 1
    
    return matrix, vocab


corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat chased the dog",
]

bow_matrix, vocab = build_bow(corpus)
print("Vocabulary:", vocab)
print()

# Display as a DataFrame for readability
vocab_sorted = sorted(vocab.keys(), key=lambda w: vocab[w])
df = pd.DataFrame(bow_matrix, columns=vocab_sorted, index=[f"Doc {i}" for i in range(len(corpus))])
print("Document-Term Matrix:")
print(df)
print()

# Demonstrate word-order blindness
doc_a = "dog bites man"
doc_b = "man bites dog"
mat_ab, vocab_ab = build_bow([doc_a, doc_b])
print(f"'{doc_a}' BoW: {mat_ab[0]}")
print(f"'{doc_b}' BoW: {mat_ab[1]}")
print(f"Identical? {np.array_equal(mat_ab[0], mat_ab[1])} <-- BoW loses word order!")

## 3. TF-IDF

### Problem with Raw Counts

Common words like "the", "is", "and" dominate BoW vectors but carry little meaning.

### Solution: TF-IDF

**TF-IDF** (Term Frequency - Inverse Document Frequency) weighs each word by:

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\frac{N}{\text{DF}(t)}$$

where:

- $\text{TF}(t, d)$ = frequency of term $t$ in document $d$ (or raw count)
- $N$ = total number of documents
- $\text{DF}(t)$ = number of documents containing term $t$
- $\log\frac{N}{\text{DF}(t)}$ = **inverse document frequency** (IDF)

**Intuition:**
- Words appearing in **many** documents get **low** IDF (e.g., "the" appears everywhere -> low weight).
- Words appearing in **few** documents get **high** IDF (e.g., "quantum" in a general corpus -> high weight).
- Result: rare, distinctive words matter more.

In [ ]:
# ---- TF-IDF from scratch ----

def compute_tfidf(documents: list) -> tuple:
    """Compute TF-IDF matrix from scratch.
    
    Returns:
        (tfidf_matrix, vocabulary)
    """
    # Build vocabulary and BoW
    bow_matrix, vocab = build_bow(documents)
    N = len(documents)
    
    # TF: raw counts (already in bow_matrix)
    tf = bow_matrix.astype(float)
    
    # DF: number of documents each word appears in
    df = (bow_matrix > 0).sum(axis=0).astype(float)
    
    # IDF: log(N / DF)
    idf = np.log(N / (df + 1e-10))  # add epsilon to avoid division by zero
    
    # TF-IDF
    tfidf = tf * idf
    
    return tfidf, vocab, idf


tfidf_matrix, vocab, idf_values = compute_tfidf(corpus)

# Show IDF values
print("IDF values (higher = more distinctive):")
for word, idx in sorted(vocab.items(), key=lambda x: x[1]):
    print(f"  {word:10s} IDF = {idf_values[idx]:.4f}")

print()
print("TF-IDF Matrix:")
vocab_sorted = sorted(vocab.keys(), key=lambda w: vocab[w])
df_tfidf = pd.DataFrame(
    np.round(tfidf_matrix, 3),
    columns=vocab_sorted,
    index=[f"Doc {i}" for i in range(len(corpus))]
)
print(df_tfidf)
print()
print("Notice: 'the' (appears in all docs) has the lowest TF-IDF weight.")
print("Distinctive words like 'mat', 'log', 'chased' have higher weights.")

## 4. Word Embeddings (Overview)

### Limitations of BoW / TF-IDF

Both BoW and TF-IDF produce **sparse, high-dimensional** vectors with **no semantic understanding**:
- `cosine_sim("happy", "joyful")` = 0 in BoW (orthogonal vectors)

### Word Embeddings: Dense, Learned Representations

Word embeddings map each word to a **dense vector** (typically 50-300 dimensions) where **semantically similar words are close together**.

### Word2Vec (Mikolov et al., 2013)

Two architectures:

- **CBOW (Continuous Bag-of-Words):** Predict the center word from surrounding context words.
  
  $$P(w_t | w_{t-k}, \ldots, w_{t+k})$$

- **Skip-gram:** Predict context words from the center word.
  
  $$P(w_{t+j} | w_t) \quad \text{for } j \in [-k, k], j \neq 0$$

### GloVe (Pennington et al., 2014)

- Uses **global co-occurrence statistics** rather than local context windows.
- Factorizes the log of the word co-occurrence matrix.
- Often performs comparably to Word2Vec.

### Pre-trained Embeddings

Instead of training from scratch, use embeddings pre-trained on large corpora:
- Word2Vec (Google News, 300d)
- GloVe (Wikipedia + Gigaword, 50d/100d/200d/300d)
- FastText (subword-aware, handles OOV better)

In [ ]:
# ---- Simulated word embeddings to illustrate the concept ----
# (Real embeddings would come from Word2Vec/GloVe pre-trained models)

np.random.seed(42)

# Simulate 5-dimensional embeddings for illustration
embeddings = {
    "king":   np.array([ 0.9,  0.8,  0.1,  0.3,  0.7]),
    "queen":  np.array([ 0.9,  0.8,  0.1,  0.3, -0.7]),
    "man":    np.array([ 0.2,  0.1,  0.0,  0.3,  0.7]),
    "woman":  np.array([ 0.2,  0.1,  0.0,  0.3, -0.7]),
    "happy":  np.array([-0.5, -0.3,  0.8,  0.6,  0.0]),
    "joyful": np.array([-0.5, -0.2,  0.9,  0.5,  0.0]),
    "sad":    np.array([-0.5, -0.3, -0.8, -0.6,  0.0]),
    "cat":    np.array([ 0.0,  0.5, -0.2,  0.8,  0.1]),
}

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Semantic similarity
print("Cosine similarities (simulated embeddings):")
pairs = [("happy", "joyful"), ("happy", "sad"), ("king", "queen"), ("king", "cat")]
for w1, w2 in pairs:
    sim = cosine_similarity(embeddings[w1], embeddings[w2])
    print(f"  sim({w1:8s}, {w2:8s}) = {sim:.4f}")

# Classic analogy: king - man + woman ≈ queen
result = embeddings["king"] - embeddings["man"] + embeddings["woman"]
print(f"\nking - man + woman = {result}")
print(f"queen              = {embeddings['queen']}")
print(f"Similarity: {cosine_similarity(result, embeddings['queen']):.4f}")

## 5. Comparison Table

| Feature | **Bag-of-Words** | **TF-IDF** | **Word Embeddings** |
|---------|:----------------:|:----------:|:-------------------:|
| **Vector type** | Sparse, high-dim | Sparse, high-dim | Dense, low-dim (50-300d) |
| **Word order** | Lost | Lost | Partially captured (via context) |
| **Semantics** | None | None | Captured |
| **OOV handling** | Ignored | Ignored | Possible (FastText) |
| **Training data** | None needed | None needed | Large corpus needed |
| **Computation** | Fast | Fast | Slow to train, fast to use |
| **Interpretability** | High (counts) | High (weights) | Low (dense vectors) |
| **Best for** | Simple baselines, small data | Better baselines | Deep learning, transfer learning |
| **Handles synonyms** | No | No | Yes |

## 6. Code: BoW with CountVectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "The cat sat on the mat",
    "The dog sat on the log",
    "The cat chased the dog",
    "Dogs and cats are common pets",
]

# Fit CountVectorizer
vectorizer = CountVectorizer(lowercase=True)
bow_sklearn = vectorizer.fit_transform(corpus)

print(f"Vocabulary: {vectorizer.get_feature_names_out()}")
print(f"Matrix shape: {bow_sklearn.shape}  (docs x vocab)")
print(f"Matrix type: {type(bow_sklearn)}  (sparse!)")
print()

# Display as dense DataFrame
df_bow = pd.DataFrame(
    bow_sklearn.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"Doc {i}" for i in range(len(corpus))]
)
print(df_bow)

In [ ]:
# ---- N-gram BoW: capture some word order ----

bigram_vectorizer = CountVectorizer(ngram_range=(1, 2), max_features=20)
bow_bigram = bigram_vectorizer.fit_transform(corpus)

print("Unigram + Bigram features:")
print(bigram_vectorizer.get_feature_names_out())
print(f"\nShape: {bow_bigram.shape}")

## 7. Code: TF-IDF with TfidfVectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(lowercase=True)
tfidf_sklearn = tfidf_vectorizer.fit_transform(corpus)

df_tfidf_sk = pd.DataFrame(
    np.round(tfidf_sklearn.toarray(), 3),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i}" for i in range(len(corpus))]
)
print("TF-IDF Matrix (sklearn):")
print(df_tfidf_sk)
print()
print("Note: sklearn uses a slightly different formula (smooth IDF, L2 normalization).")
print("  sklearn IDF: log((1+N)/(1+DF)) + 1, then L2-normalize each row.")

In [ ]:
# ---- Document similarity with TF-IDF ----

from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

sim_matrix = sklearn_cosine(tfidf_sklearn)
print("Document Similarity Matrix (cosine on TF-IDF):")
print(pd.DataFrame(
    np.round(sim_matrix, 3),
    index=[f"Doc {i}" for i in range(len(corpus))],
    columns=[f"Doc {i}" for i in range(len(corpus))]
))
print()
print("Doc 0 and Doc 1 are most similar (both about an animal sitting on something).")

## 8. Code: Text Classification Comparison

We compare BoW, TF-IDF, and simple embedding-based features on a small text classification task.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Small sentiment dataset
texts = [
    "This movie is great and wonderful",
    "Excellent film with amazing performances",
    "I loved this movie so much",
    "Best movie I have ever seen",
    "Fantastic story and brilliant acting",
    "Really enjoyed this film a lot",
    "Outstanding movie with great direction",
    "Wonderful performances throughout the film",
    "This movie is terrible and boring",
    "Worst film ever made and awful",
    "I hated this movie completely",
    "Terrible acting and bad story",
    "Boring and dull waste of time",
    "Really disliked this film very much",
    "Awful movie with poor direction",
    "Disappointing performances throughout the film",
]
labels = np.array([1]*8 + [0]*8)  # 1=positive, 0=negative

# ---- Method 1: BoW + Logistic Regression ----
bow_pipeline = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("classifier", LogisticRegression(random_state=42, max_iter=1000)),
])

# ---- Method 2: TF-IDF + Logistic Regression ----
tfidf_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("classifier", LogisticRegression(random_state=42, max_iter=1000)),
])

# ---- Method 3: Simple embedding average ----
# Build simple word vectors from co-occurrence (simulated for small data)
# In practice, you would use pre-trained Word2Vec/GloVe/FastText

# Use TF-IDF-weighted average of random vectors as a stand-in
np.random.seed(42)
all_words = set(" ".join(texts).lower().split())
word_vectors = {w: np.random.randn(50) for w in all_words}

def texts_to_embedding_features(texts, word_vecs, dim=50):
    """Average word vectors for each document."""
    features = np.zeros((len(texts), dim))
    for i, text in enumerate(texts):
        words = text.lower().split()
        vecs = [word_vecs[w] for w in words if w in word_vecs]
        if vecs:
            features[i] = np.mean(vecs, axis=0)
    return features

X_emb = texts_to_embedding_features(texts, word_vectors)
emb_clf = LogisticRegression(random_state=42, max_iter=1000)

# ---- Evaluate all three ----
print("Text Classification: BoW vs TF-IDF vs Embeddings")
print("=" * 55)

for name, pipeline in [("BoW", bow_pipeline), ("TF-IDF", tfidf_pipeline)]:
    scores = cross_val_score(pipeline, texts, labels, cv=4, scoring="accuracy")
    print(f"{name:12s}  Accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

scores_emb = cross_val_score(emb_clf, X_emb, labels, cv=4, scoring="accuracy")
print(f"{'Embeddings':12s}  Accuracy: {scores_emb.mean():.3f} (+/- {scores_emb.std():.3f})")

print()
print("Note: With this tiny dataset, BoW/TF-IDF often win because embeddings")
print("need large corpora to learn good representations. On large datasets,")
print("pre-trained embeddings typically outperform sparse methods.")

## 9. Common Mistakes & Debugging Tips

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Confusing "bagging" with "Bag-of-Words" | Using ensemble methods when text representation was needed | Remember: "bagging" in NLP = Bag-of-Words |
| Not lowercasing before BoW/TF-IDF | "The" and "the" counted separately | Use `lowercase=True` in vectorizer |
| Using BoW without stop-word removal | Common words dominate | Use `stop_words='english'` or TF-IDF |
| Too large vocabulary | Memory issues, noise | Set `max_features` or `min_df`/`max_df` |
| Using `.toarray()` on large sparse matrices | Memory crash | Work with sparse matrices directly |
| Averaging embeddings without weighting | Common words dominate | Use TF-IDF-weighted averaging |
| Fitting vectorizer on test data | Data leakage | `fit` on train only, `transform` on test |

## 10. Exercises

### Exercise 1: BoW vs TF-IDF Comparison

Given the corpus below, compute both BoW and TF-IDF representations, then determine which document pair is most similar under each method.

In [ ]:
exercise_corpus = [
    "machine learning is a subset of artificial intelligence",
    "deep learning is a subset of machine learning",
    "natural language processing uses machine learning",
    "the weather today is sunny and warm",
]

# TODO:
# 1. Compute BoW and TF-IDF representations
# 2. Compute cosine similarity between all document pairs
# 3. Find the most similar pair for each method
# Your code here

### Exercise 2: Vocabulary Size Impact

Experiment with different `max_features` values for CountVectorizer and observe the effect on classification accuracy.

In [ ]:
# TODO:
# 1. Try max_features = [5, 10, 20, 50, None]
# 2. For each, run cross-validation on the sentiment dataset above
# 3. Report accuracy for each setting
# Your code here

## 11. Exercise Solutions

In [ ]:
# ---- Exercise 1 Solution ----

from sklearn.metrics.pairwise import cosine_similarity as cs

exercise_corpus = [
    "machine learning is a subset of artificial intelligence",
    "deep learning is a subset of machine learning",
    "natural language processing uses machine learning",
    "the weather today is sunny and warm",
]

# BoW
cv = CountVectorizer()
bow_ex = cv.fit_transform(exercise_corpus)
bow_sim = cs(bow_ex)

# TF-IDF
tv = TfidfVectorizer()
tfidf_ex = tv.fit_transform(exercise_corpus)
tfidf_sim = cs(tfidf_ex)

print("BoW Similarity Matrix:")
print(pd.DataFrame(np.round(bow_sim, 3),
    index=[f"Doc {i}" for i in range(4)],
    columns=[f"Doc {i}" for i in range(4)]))
print()

print("TF-IDF Similarity Matrix:")
print(pd.DataFrame(np.round(tfidf_sim, 3),
    index=[f"Doc {i}" for i in range(4)],
    columns=[f"Doc {i}" for i in range(4)]))
print()

# Find most similar pairs (exclude diagonal)
for name, sim_mat in [("BoW", bow_sim), ("TF-IDF", tfidf_sim)]:
    np.fill_diagonal(sim_mat, 0)
    i, j = np.unravel_index(sim_mat.argmax(), sim_mat.shape)
    print(f"{name}: Most similar pair = Doc {i} & Doc {j} (sim={sim_mat[i,j]:.3f})")

In [ ]:
# ---- Exercise 2 Solution ----

print("\nVocabulary Size Impact on Classification:")
print("=" * 45)

for max_feat in [5, 10, 20, 50, None]:
    pipe = Pipeline([
        ("vec", CountVectorizer(max_features=max_feat)),
        ("clf", LogisticRegression(random_state=42, max_iter=1000)),
    ])
    scores = cross_val_score(pipe, texts, labels, cv=4, scoring="accuracy")
    feat_str = str(max_feat) if max_feat else "All"
    print(f"  max_features={feat_str:5s}  Accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")